# 3. Analizzare un documento di due pagine

Analizza il conto di Mario Rossi senza duplicare la conferma di pagamento della seconda pagina. Risultato atteso: USD 1587.10, saldo USD 0.

Documento: [esercizio3.pdf](../file/esercizio3.pdf).

Soluzione completa: esegui le celle in ordine con il kernel del venv. Ogni notebook parte da zero: nessun `main()`, `%run` o stato condiviso con gli altri esercizi.

[Traccia da svolgere](../esercizi/03-analisi.ipynb).

## 1. Import e configurazione

Esegui dalla cartella `soluzioni`. `load_dotenv(r"..\.env", override=True)` carica il file noto e applica i suoi valori. I risultati vengono soltanto stampati.

In [1]:
import json
import os
from pathlib import Path
from azure.ai.contentunderstanding import ContentUnderstandingClient
from azure.identity import DefaultAzureCredential
from dotenv import load_dotenv

In [2]:
load_dotenv(r"..\.env", override=True)
DOCUMENTO = Path(r"..\file\esercizio3.pdf")
document_bytes = DOCUMENTO.read_bytes()

## 2. Client ed esecuzione

`DefaultAzureCredential()` usa l'identita Azure disponibile (in locale: `az login`). `ContentUnderstandingClient` riceve endpoint e credenziale.

`begin_analyze_binary` riceve ID analyzer e byte PDF; `result()` attende il completamento. Il blocco `finally` chiude client e credenziale.

In [3]:
credential = DefaultAzureCredential()
# Crea il client usando endpoint CU e credenziale Azure corrente.
client = ContentUnderstandingClient(
    endpoint=os.environ["AZURE_CONTENTUNDERSTANDING_ENDPOINT"],
    credential=credential,
)
try:
    # Invia i byte del PDF all'analyzer prebuilt e attende il risultato.
    result = client.begin_analyze_binary(
        analyzer_id="prebuilt-receipt.hotel",
        binary_input=document_bytes,
    ).result()
finally:
    client.close()
    credential.close()

## 3. Osserva il risultato

Mostra il risultato della chiamata Azure. `as_dict()` converte gli oggetti SDK in dizionari; `json.dumps(..., ensure_ascii=False, indent=2)` produce JSON leggibile. `print` mostra il risultato senza salvarlo su disco.

In [4]:
for content in result.contents:
    print(content.markdown)
    fields = content.as_dict()["fields"]
    print(json.dumps(fields, ensure_ascii=False, indent=2))

# HOTEL ESEMPIO

Guest folio / Ricevuta soggiorno | H-3001

123 Example Avenue, Seattle, WA 98101, USA

Guest / Ospite: Mario Rossi
Room / Camera: 204
Arrival: 2026-09-01 Departure: 2026-09-04

Currency: USD | Folio status: CLOSED


## Charges and payments


<table>
<tr>
<th>Date</th>
<th>Description</th>
<th>Charges USD</th>
<th>Credits USD</th>
</tr>
<tr>
<td>2026-09-01</td>
<td>Room accommodation</td>
<td>500.00</td>
<td></td>
</tr>
<tr>
<td>2026-09-02</td>
<td>Room accommodation</td>
<td>500.00</td>
<td></td>
</tr>
<tr>
<td>2026-09-03</td>
<td>Room accommodation</td>
<td>500.00</td>
<td></td>
</tr>
<tr>
<td>2026-09-04</td>
<td>Occupancy tax</td>
<td>87.10</td>
<td></td>
</tr>
<tr>
<td>2026-09-04</td>
<td>Card payment</td>
<td></td>
<td>1587.10</td>
</tr>
</table>


Total charges: USD 1587.10

Total payments: USD 1587.10

Balance due: USD 0.00 - PAID IN FULL

Continued on the next page: settlement details for the same folio.

<!-- PageFooter: DATI SINTETICI / TRAINING SAMPLE - Non v